In [ ]:
# Import standard packages
import numpy as np
import pandas as pd
import itertools
import os
from scipy.signal import butter, filtfilt
from scipy.stats import gaussian_kde
from sklearn.preprocessing import MinMaxScaler
from scipy.interpolate import interp1d

# Import plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from scipy.interpolate import interp1d


In [ ]:
directory_name = input('CSV File Directory:')

In [ ]:
# Navigate to the csv files directory
os.chdir(directory_name)

In [ ]:
# current directory csv files
csvs = [x for x in os.listdir(directory_name) if x.endswith('.csv')]
# stats.csv -> stats
fns = [os.path.splitext(os.path.basename(x))[0] for x in csvs]

dic_csv = {}
for i in range(len(fns)):
    dic_csv[fns[i]] = pd.read_csv(csvs[i])

In [ ]:
clean_dic={}

In [ ]:
#dic_csv['01_P1_20_ASP100_WDP50_18Jul24'].columns

#for key, df in dic_csv.items():
    #parameter_name = df.iloc[:,6].name

#for key, df in dic_csv.items():
    #df.rename(columns={'P3-Air-Supply-pressure':'P3-Air Suction Pressure'},inplace=True)

In [ ]:
#Collect all columname in the list
for df in dic_csv.values():
    parametername_list=list(set(df.columns))
#Identify the correct parameter name from the list
if 'P1-Air-Supply-pressure' in parametername_list:
    parameter_name= 'P1-Air-Supply-pressure'
elif 'P2-Air-Supply-pressure' in parametername_list:
    parameter_name='P2-Air-Supply-pressure'
elif 'P3-Air Suction Pressure' in parametername_list:
    parameter_name='P3-Air Suction Pressure'
else:
    parameter_name=None
#Print the parameter name
print(parameter_name)

In [ ]:
for key, df in dic_csv.items():
    clean_dic[key] = df[df[parameter_name] > 80]

In [ ]:
for key, df in clean_dic.items():
    df.reset_index(inplace=True, drop=True)

In [ ]:
# Generate the list of date elements based on the number of keys
Month_date_list = pd.date_range('2024-04-11 00:00:00.000', freq='1D', periods=len(clean_dic)).strftime('%Y-%m-%d %H:%M:%S.%f').tolist()

In [ ]:
for (key, df), i in zip(clean_dic.items(),Month_date_list) :    
    df['newtime'] = pd.date_range(i, freq='30ms', periods=len(df))

for key, df in clean_dic.items():
    df['newtime'] =pd.to_datetime(df['newtime'])
    
for key, df in clean_dic.items():
    df.set_index(['newtime'], inplace=True)

In [ ]:
total_hrs=0
for key,df in clean_dic.items():
   hrs=np.round((((df.index[-1]-df.index[0]).total_seconds())/3600),decimals=1)
   total_hrs=total_hrs+hrs

In [ ]:
total_hrs

In [ ]:
# Function to apply a high-pass filter
def high_pass_filter(data, cutoff, fs, order=5):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    filtered_data = filtfilt(b, a, data)
    return filtered_data

In [ ]:

#temp_dic = {}
Freq_list = [] 
Amp_list = []
result_df = pd.DataFrame(columns=['Freq','Amp'])

for key, df in clean_dic.items():
    # Define the time interval for slicing
    start_time = df.index[0]
    end_time = df.index[-1]
    time_interval = pd.Timedelta(minutes=1)
    
    current_time = start_time
    while current_time <= end_time:
        
        next_time = current_time + time_interval
        
        # Slice the DataFrame based on the specified time range
        sliced_df = df.iloc[(df.index >= current_time) & (df.index <= next_time)]
        
        # Perform FFT analysis only if there is data in the sliced DataFrame
        n = len(sliced_df)
        if n > 0:
            t = sliced_df.index
            s = sliced_df['Right-Vibration-Data']

            # Apply high-pass filter
            dt = (t[1] - t[0]).total_seconds()  # Sampling interval in seconds
            fs = 1 / dt  # Sampling frequency in Hz
            cutoff_frequency = 0.1  # Set the cutoff frequency for the high-pass filter in Hz
            filtered_s = high_pass_filter(s, cutoff=cutoff_frequency, fs=fs)

            # FFT
            filtered_s -= filtered_s.mean()
            fr = np.fft.rfftfreq(n, dt)
            fou = np.fft.rfft(filtered_s)
            
            # Create a DataFrame for the current fr,amp
            temp_df = pd.DataFrame({'Frequency': fr,'Amplitude': np.abs(fou)})

            #Filter max apllitude frim temp_df and put it in to df_filter
            df_Filter = temp_df.loc[temp_df['Amplitude'].idxmax()]
            
            #Append coresponding maximum amplitude frequency values in to freq_amp list
            Freq_list.append((df_Filter['Frequency']))
            Amp_list.append((df_Filter['Amplitude']))            
        else:
            pass
        current_time = next_time    
    #temp_dic[key] = result_df
# Append the DataFrame to the list
result_df = pd.DataFrame({'Freq':Freq_list,'Amp':Amp_list})

In [ ]:
#Filter anamalies from the data frame
result_df = result_df[(result_df['Freq']>0)&(result_df['Freq']<2.2)]

In [ ]:
#Reset the data frame
result_df.reset_index(drop=True,inplace=True)

In [ ]:
#Normalize the data frame values
normalized_variables = MinMaxScaler().fit_transform(result_df)

#Insert the normalized vaules in to the new columns
for value_index,col_name in enumerate(result_df.columns):
    result_df["Norm_{}".format(col_name)] = normalized_variables[:,value_index]
    
#Create Diaprhagm liftime
timeline = [n for n in np.linspace(0,total_hrs,len(result_df))]

#Insert timeline in to dataframe
result_df['Diaphragm_life'] = timeline

#Set Diaphragm life as index
result_df.set_index(['Diaphragm_life'],inplace=True)

In [ ]:
def plot_funtion(dataframe):
    a = len(dataframe.columns)  # number of rows
    b = 1  # number of columns
    c = 1  # initialize plot counter
    fig1 = plt.figure(figsize=(35,20))
       
    for i in dataframe.columns:
        # Create a subplot and save the plot
        plt.subplot(a, b, c)
        plt.plot(dataframe.index, dataframe[i],marker = '.')
        plt.xticks(np.arange(np.min(dataframe.index),np.max((dataframe.index)),0.1))             
        #plt.ylim(0,0.5)
        plt.title(f"{i}{' vs Diaphragm Life Time'}")
        #plt.legend(df.columns, loc='lower center', bbox_to_anchor=(0.5, -0.14), ncol=8)
        c = c + 1
    return plot_funtion

In [ ]:
result_df.head()

In [ ]:
plot_funtion(result_df)

In [ ]:
plt.figure(figsize=(20,10))
plt.subplot(4, 1, 1)
plt.plot(result_df.index,result_df.iloc[:,0])
plt.xticks(np.arange(np.min(result_df.index),np.max((result_df.index)),3))
#plt.ylim(0,2)
plt.subplot(4, 1, 2)
plt.plot(result_df.index,result_df.iloc[:,1])
plt.xticks(np.arange(np.min(result_df.index),np.max((result_df.index)),3))
#plt.ylim(0,2)
plt.subplot(4, 1, 3)
plt.plot(result_df.index,result_df.iloc[:,2])
plt.xticks(np.arange(np.min(result_df.index),np.max((result_df.index)),3))
plt.subplot(4, 1, 4)
plt.plot(result_df.index,result_df.iloc[:,3])
plt.xticks(np.arange(np.min(result_df.index),np.max((result_df.index)),3))
#plt.legend(labels=(result_df.iloc[:,2].name,result_df.iloc[:,5].name), loc='lower center', bbox_to_anchor=(0.5, -0.12), ncol=8)
#plt.xlim(161,162)
plt.show()

In [ ]:
result_df

In [ ]:
#Read frequency  data frame as csv
#output_files = os.path.join(directory_name,'T20_20_D4_Df0_freq&Amp.csv')
#result_df.to_csv(output_files)

In [ ]:
# set figure size 
plt.figure(figsize = (30,6)) 
#plt.yticks(np.arange(np.min(Df0_Freq_df['Frequency']),np.max((Df0_Freq_df['Frequency'])),0.5))
sns.regplot(x=result_df['Diaphragm_life'],y=result_df['Amp'], lowess=True, line_kws={'color':'red'})
#plt.ylim(0,1.5)

In [ ]:
# set figure size 
plt.figure( figsize = (30,6)) 
#plt.yticks(np.arange(np.min(Df0_Freq_df['Frequency']),np.max((Df0_Freq_df['Frequency'])),0.5))
sns.regplot(x=result_df['Diaphragm_life'], y=result_df['Norm_Freq'], lowess=True, line_kws={'color':'red'})
#plt.ylim(0,1.5)

In [ ]:
f = interp1d(x=result_df.index,y=result_df['Norm_Freq'],kind=7)
x2 = np.linspace(start=0,stop=135.9,num=10000)
y2 = f(x2)

In [ ]:
plt.figure(figsize=(25,5))
plt.plot(x2, y2, color='b')
plt.plot(result_df.index,result_df['Norm_Freq'], ls='', marker='.', color='r')
plt.xlabel('Hrs')
plt.show()

In [ ]:
f1 = interp1d(x=Norm_df.index,y=Norm_df['Amp'],kind=2)
x3 = np.linspace(start=0,stop=278.05,num=10000)
y3 = f1(x3)

In [ ]:
plt.figure(figsize=(25,5))
plt.plot(x3, y3, color='b')
plt.plot(Norm_df.index,Norm_df['Amp'], ls='', marker='o', color='r')
plt.xlabel('Hrs')
plt.show()

In [ ]:
'''
plt.figure(figsize=(50,10))
plt.plot(merged_df.index,merged_df.iloc[:,2],color='magenta')
plt.plot(merged_df.index,merged_df.iloc[:,5],color='blue')
plt.xlabel('Hrs')
plt.ylabel('Data')
plt.title(f'{parameter_name[:2]}-{"ASP,CPM"}')
plt.legend(labels=(merged_df.iloc[:,2].name,merged_df.iloc[:,5].name), loc='lower center', bbox_to_anchor=(0.5, -0.22), ncol=8)
plt.ylim(0,150)
#plt.xlim(161,162)
plt.show()
'''

In [ ]:
'''
def max_freq(lst):
    max_values = []
    for i in range(0, len(lst), 2):
        # Select a slice of the list with up to 5 elements
        five_elements = lst[i:i+2]
        # Find the maximum value in this slice
        max_value = np.average(five_elements)
        max_values.append(max_value)
    return max_values

# Example usage
max_freq_list = max_freq(freq_amp_dfs_Sf1)

#filename = os.path.join(directory_name, f'1.P2_15_10Hrs_Freq_FFT.png')
#plt.savefig(filename)
#plt.close()

xi = np.linspace(0, np.max(temp_df['Frequency']), 80000)

kde = gaussian_kde(temp_df['Frequency'], weights = temp_df['Amplitude'], bw_method = 0.002)  #tune `bw_method` to get the bandwidth you want
plt.figure(figsize=(20,5))

plt.subplot(211)
plt.plot(temp_df['Frequency'], temp_df['Amplitude'])

plt.subplot(212)
plt.plot(xi, kde.pdf(xi))
#plt.xlim(0,18)
'''

In [ ]:
for key, df in clean_dic.items():
    # Define the time interval for slicing
    start_time = df.index[0]
    end_time = df.index[-1]
    time_interval = pd.Timedelta(minutes=5)
    
    current_time = start_time
    while current_time <= end_time:
        
        next_time = current_time + time_interval
        
        # Slice the DataFrame based on the specified time range
        sliced_df = df.iloc[(df.index >= current_time) & (df.index <= next_time)]
        
        # Perform FFT analysis only if there is data in the sliced DataFrame
        n = len(sliced_df)
        if n > 0:
            t = sliced_df.index
            s = sliced_df['Right-Vibration-Data']

            # Apply high-pass filter
            dt = (t[1] - t[0]).total_seconds()  # Sampling interval in seconds
            fs = 1 / dt  # Sampling frequency in Hz
            cutoff_frequency = 0.1  # Set the cutoff frequency for the high-pass filter in Hz
            filtered_s = high_pass_filter(s, cutoff=cutoff_frequency, fs=fs)

            # FFT
            filtered_s -= filtered_s.mean()
            fr = np.fft.rfftfreq(n, dt)
            fou = np.fft.rfft(filtered_s)
            
            # Plot and save the FFT plot
            plt.figure(figsize=(30,8))
            plt.subplot(211)
            plt.plot(t, filtered_s,color='blue',linewidth=1)
            plt.title('Data with Noise')

            plt.subplot(212)
            plt.plot(np.fft.fftshift(fr), np.fft.fftshift(np.abs(fou)),color='blue',linewidth=1)
            plt.xticks(np.arange(min(np.fft.fftshift(fr)), max(np.fft.fftshift(fr)), 0.4))
            #plt.ylim(0, 6000)
            plt.title('Spectrum of Noisy Data')

            # Save the plot with a meaningful filename
            filename = os.path.join(directory_name, f'{key}_{current_time.strftime("%H-%M-%S")}_{next_time.strftime("%H-%M-%S")}_FFT.png')
            plt.savefig(filename)
            plt.close() 
        else:
            pass
        current_time = next_time   